<a href="https://colab.research.google.com/github/mohamedalaaaz/testpytroch/blob/main/Al%20home%20.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")

text = "3 bedroom house with a large kitchen"
inputs = tokenizer(text, return_tensors="pt")
outputs = model(**inputs)
text_embedding = outputs.last_hidden_state.mean(dim=1)

In [ ]:
import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, text_embedding_dim, output_dim):
        super(Generator, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(text_embedding_dim, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim),  # e.g. flattened image
            nn.Tanh()
        )

    def forward(self, text_embedding):
        return self.fc(text_embedding)

In [ ]:
for epoch in range(num_epochs):
    for text, image in dataloader:
        text_embed = text_encoder(text)
        generated_image = generator(text_embed)
        loss = loss_fn(generated_image, real_image)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

In [ ]:

import torch.nn as nn

class Generator(nn.Module):
    def __init__(self, input_dim=128, output_dim=8):
        super(Generator, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, output_dim),
        )

    def forward(self, x):
        return self.fc(x)

In [ ]:
import torch
from transformers import BertTokenizer, BertModel

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert = BertModel.from_pretrained("bert-base-uncased")

def encode_text(text_list):
    inputs = tokenizer(text_list, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
        outputs = bert(**inputs)
    return outputs.last_hidden_state.mean(dim=1)  # (batch_size, hidden_dim)

In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from model.generator import Generator
from utils import encode_text

class LayoutDataset(Dataset):
    def __init__(self, csv_path):
        df = pd.read_csv(csv_path)
        self.texts = df['description'].tolist()
        self.layouts = [eval(x) for x in df['layout'].tolist()]

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return self.texts[idx], torch.tensor(self.layouts[idx], dtype=torch.float)

# Load data
dataset = LayoutDataset("data/samples.csv")
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# Model setup
model = Generator()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.MSELoss()

# Training loop
for epoch in range(20):
    total_loss = 0
    for texts, targets in loader:
        embeddings = encode_text(texts)
        outputs = model(embeddings)
        loss = loss_fn(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")